In [1]:
import os
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm
 
from scipy.stats import norm
from scipy.optimize import minimize

In [2]:
# ----------------------------- config ---------------------------------------
EPS = 1e-6
MIN_SAMPLES = 10
N_DRESS = 50
RNG_SEED = 42

years_train = np.arange(1959, 2000)
years_verif = np.arange(2000, 2021)
T_CENTER = float(years_train.mean())   # e.g., 1979.0 for 1959..1999

# all Gaussian-distributed variables to post-process
vars_keep = [
    "Fort_Bragg_SPEI_09_mean",         # ACC +0.04, but CRPSS 0.24 -> -0.11
    "Fort_Bragg_SPEI_48_mean",         # ACC +0.04, but CRPSS -0.93 (!)
    "Fort_Bragg_SPEI_48_min",          # ACC +0.03, but CRPSS -1.41 (!)
    "Fort_Bragg_simple_T2_1d_max",     # both poor; trend less bad on both
    "Fort_Bragg_simple_T2_1h_max",
    "Fort_Bragg_simple_T2_30d_max",    # mild, both metrics agree
    "Guam_SPEI_03_mean",
    "Guam_SPEI_03_min",
    "Guam_SPEI_48_mean",               # clear win on both metrics
    "Guam_SPEI_48_min",                # clear win on both metrics
    "Guam_simple_T2_1h_max",           # mild, both metrics agree
    "Pituffik_simple_T2_1d_min",       # clear win on both metrics
    "Pituffik_simple_T2_1h_min",       # clear win on both metrics
    "Yuma_PG_SPEI_48_mean",            # ACC +0.07, CRPS worse
    "Yuma_PG_SPEI_48_min",             # ACC +0.09, CRPS worse
    "Yuma_PG_simple_T2_1d_mean",       # ACC +0.04, CRPS mildly worse
    "Yuma_PG_simple_T2_1d_min",        # ACC +0.05, CRPS clearly worse
]

data_dir = '/glade/derecho/scratch/ksha/EPRI_data/METRICS/'
in_input  = os.path.join(data_dir, 'STN_CESM_ALL_20260604.zarr')
in_target = os.path.join(data_dir, 'STN_ERA5_ALL_20260604.zarr')

# ----------------------------- load -----------------------------------------
ds_input = xr.open_zarr(in_input)[vars_keep].drop_vars('valid_year', errors='ignore')
ds_input_train = ds_input.sel(init_year=years_train)
ds_input_verif = ds_input.sel(init_year=years_verif)
 
ds_target = xr.open_zarr(in_target)[vars_keep]
 
n_member = ds_input_train.sizes['member']
n_lead   = ds_input_train.sizes['lead_year']
n_train  = ds_input_train.sizes['init_year']
n_verif  = ds_input_verif.sizes['init_year']
 
lead_years        = ds_input_train['lead_year'].values
init_years_train  = ds_input_train['init_year'].values
init_years_verif  = ds_input_verif['init_year'].values

In [3]:
# ============================================================================
# Gaussian EMOS
# ============================================================================
def crps_gaussian(mu, sigma, y):
    """Closed-form CRPS for N(mu, sigma^2)."""
    sigma = np.maximum(sigma, EPS)
    z = (y - mu) / sigma
    return sigma * (z * (2.0 * norm.cdf(z) - 1.0)
                    + 2.0 * norm.pdf(z)
                    - 1.0 / np.sqrt(np.pi))
 
def emos_objective(params, xbar, sx2, y):
    a, b, c, d = params
    mu = a + b * xbar
    sigma2 = c * c + d * d * sx2
    sigma = np.sqrt(np.maximum(sigma2, EPS * EPS))
    return float(np.mean(crps_gaussian(mu, sigma, y)))

def emos_objective_trend(params, xbar, sx2, t, y):
    """Mean CRPS for the trend-augmented Gaussian EMOS:
       mu      = a + b * xbar + e * t
       sigma^2 = c^2 + d^2 * sx2
    """
    a, b, c, d, e = params
    mu = a + b * xbar + e * t
    sigma2 = c * c + d * d * sx2
    sigma = np.sqrt(np.maximum(sigma2, EPS * EPS))
    return float(np.mean(crps_gaussian(mu, sigma, y)))


def fit_emos(xbar, sx2, y, init_year, gamma_min=0.1):
    """CRPS-minimised EMOS with a linear time-trend, standardized (gamma)
    parameterisation that prevents collapse to climatology.

    Internal model in standardized space (each variable centred and scaled):
      mu_s      = gamma * xs + epsilon * ts
      sigma_s^2 = c_s^2 + d_s^2 * (sx2 / xbar_std^2)

    where gamma in [gamma_min, 1.0] is the dimensionless signal-amplification
    coefficient (= corr(X_bar, y) at the OLS optimum) and epsilon is the
    dimensionless trend coefficient (free sign, no bound).

    Returns params in *original units* for downstream compatibility:
      params = [a, b, c, d, e]
    """
    valid = (np.isfinite(xbar) & np.isfinite(sx2)
             & np.isfinite(y) & np.isfinite(init_year))
    if valid.sum() < MIN_SAMPLES:
        return np.full(5, np.nan), False

    xbar_v = xbar[valid]
    sx2_v  = sx2[valid]
    y_v    = y[valid]
    t_v    = init_year[valid].astype(np.float64) - T_CENTER

    # ----- Standardization scales (training period only) ------------------
    xbar_mean = float(np.mean(xbar_v))
    xbar_std  = max(float(np.std(xbar_v, ddof=1)), EPS)
    y_mean    = float(np.mean(y_v))
    y_std     = max(float(np.std(y_v, ddof=1)),    EPS)
    t_std     = max(float(np.std(t_v, ddof=1)),    EPS)

    xs       = (xbar_v - xbar_mean) / xbar_std
    ys       = (y_v - y_mean) / y_std
    ts       = t_v / t_std
    sx2_norm = sx2_v / (xbar_std ** 2)

    # ----- OLS init in standardized space (variables already centred) -----
    try:
        A = np.column_stack([xs, ts])
        coef, *_ = np.linalg.lstsq(A, ys, rcond=None)
        gamma_0, eps_0 = float(coef[0]), float(coef[1])
    except np.linalg.LinAlgError:
        gamma_0, eps_0 = 0.5, 0.0

    # Clamp gamma to [gamma_min, 1.0]; if floored, refit epsilon alone
    if gamma_0 < gamma_min:
        gamma_0 = gamma_min
        try:
            ys_resid = ys - gamma_0 * xs
            eps_0 = float(np.sum(ys_resid * ts) / max(np.sum(ts * ts), EPS))
        except Exception:
            eps_0 = 0.0
    elif gamma_0 > 1.0:
        gamma_0 = 1.0

    # ----- Variance init from standardized residuals ----------------------
    resid2 = (ys - gamma_0 * xs - eps_0 * ts) ** 2
    try:
        B = np.column_stack([np.ones_like(sx2_norm), sx2_norm])
        cd2, *_ = np.linalg.lstsq(B, resid2, rcond=None)
        c2_0 = max(float(cd2[0]), EPS)
        d2_0 = max(float(cd2[1]), EPS)
    except np.linalg.LinAlgError:
        c2_0 = float(np.var(ys)) + EPS
        d2_0 = 0.0

    # ----- L-BFGS-B refinement -------------------------------------------
    def obj(p):
        gamma, c, d, eps = p
        mu_s    = gamma * xs + eps * ts
        sigma_s = np.sqrt(np.maximum(c * c + d * d * sx2_norm, EPS * EPS))
        return float(np.mean(crps_gaussian(mu_s, sigma_s, ys)))

    x0 = np.array([gamma_0, np.sqrt(c2_0), np.sqrt(d2_0), eps_0])
    bounds = [(gamma_min, 1.0),   # gamma: [gamma_min, 1]
              (None, None),        # c: free (squared internally)
              (None, None),        # d: free (squared internally)
              (None, None)]        # eps: trend, any sign

    res = minimize(obj, x0, method='L-BFGS-B', bounds=bounds)
    gamma_h, c_s, d_s, eps_h = (res.x if res.success else x0)

    # ----- Convert back to original units ---------------------------------
    b_hat  = gamma_h * y_std / xbar_std
    e_hat  = eps_h   * y_std / t_std
    a_hat  = y_mean  - b_hat * xbar_mean
    c_phys = c_s * y_std
    d_phys = d_s * y_std / xbar_std

    return np.array([a_hat, b_hat, c_phys, d_phys, e_hat]), bool(res.success)

def lookup_target(y_full, valid_years_needed, valid_years_avail):
    out = np.full(len(valid_years_needed), np.nan, dtype=np.float64)
    in_mask = np.isin(valid_years_needed, valid_years_avail)
    if in_mask.any():
        out[in_mask] = y_full.sel(valid_year=valid_years_needed[in_mask]).values
    return out

In [4]:
# ============================================================================
# Step 1: Training -- fit EMOS per (variable, lead_year)
# ============================================================================
# params_dict = {v: np.full((n_lead, 4), np.nan) for v in vars_keep}
params_dict = {v: np.full((n_lead, 5), np.nan) for v in vars_keep}
fit_flags   = {v: np.zeros(n_lead, dtype=bool) for v in vars_keep}
 
for var in tqdm(vars_keep, desc='EMOS training '):
    x_train = ds_input_train[var].values
    y_full  = ds_target[var]
    valid_years_avail = y_full['valid_year'].values
 
    for il, lead in enumerate(lead_years):
        xbar = x_train[:, il, :].mean(axis=1)
        sx2  = x_train[:, il, :].var(axis=1, ddof=1)
        valid_yrs_needed = init_years_train + int(lead)
        y_arr = lookup_target(y_full, valid_yrs_needed, valid_years_avail)
        # params, success = fit_emos(xbar, sx2, y_arr)
        params, success = fit_emos(xbar, sx2, y_arr, init_years_train, gamma_min=0.1)
        params_dict[var][il, :] = params
        fit_flags[var][il]      = success

EMOS training : 100%|██████████| 17/17 [00:11<00:00,  1.53it/s]


In [5]:
# ============================================================================
# Step 2: Calibration on verification data
# Step 3: Ensemble dressing (50 members)
# ============================================================================
rng = np.random.default_rng(RNG_SEED)
 
calib_data = {}
dress_data = {}
 
for var in tqdm(vars_keep, desc='EMOS calibration'):
    x_verif = ds_input_verif[var].values
    P = params_dict[var]
 
    mu_arr    = np.full((n_verif, n_lead), np.nan)
    sigma_arr = np.full((n_verif, n_lead), np.nan)
 
    # for il in range(n_lead):
    #     a, b, c, d = P[il]
    #     if not np.isfinite(a):
    #         continue
    #     xbar = x_verif[:, il, :].mean(axis=1)
    #     sx2  = x_verif[:, il, :].var(axis=1, ddof=1)
    #     mu_arr[:, il]    = a + b * xbar
    #     sigma_arr[:, il] = np.sqrt(np.maximum(c * c + d * d * sx2, EPS * EPS))

    for il in range(n_lead):
        a, b, c, d, e = P[il]
        if not np.isfinite(a):
            continue
        xbar = x_verif[:, il, :].mean(axis=1)
        sx2  = x_verif[:, il, :].var(axis=1, ddof=1)
        t_verif = init_years_verif.astype(np.float64) - T_CENTER
        mu_arr[:, il]    = a + b * xbar + e * t_verif
        sigma_arr[:, il] = np.sqrt(np.maximum(c*c + d*d*sx2, EPS*EPS))
    
    calib_data[var + '_mu']    = (('init_year', 'lead_year'), mu_arr)
    calib_data[var + '_sigma'] = (('init_year', 'lead_year'), sigma_arr)
 
    noise = rng.standard_normal((n_verif, n_lead, N_DRESS))
    dressed = mu_arr[:, :, None] + sigma_arr[:, :, None] * noise
    dress_data[var] = (('init_year', 'lead_year', 'member'), dressed)
 
ds_calib = xr.Dataset(
    calib_data,
    coords={'init_year': init_years_verif, 'lead_year': lead_years},
)
 
ds_dress = xr.Dataset(
    dress_data,
    coords={
        'init_year': init_years_verif,
        'lead_year': lead_years,
        'member':    np.arange(N_DRESS),
    },
)
 
param_arr = np.stack([params_dict[v] for v in vars_keep], axis=0)
ds_params = xr.Dataset(
    {'emos_params': (('variable', 'lead_year', 'param'), param_arr),
     'fit_success': (('variable', 'lead_year'),
                     np.stack([fit_flags[v] for v in vars_keep], axis=0))},
    coords={
        'variable':  vars_keep,
        'lead_year': lead_years,
        'param': ['a', 'b', 'c', 'd', 'e'], #'param':     ['a', 'b', 'c', 'd'],
    },
)

EMOS calibration: 100%|██████████| 17/17 [00:03<00:00,  5.55it/s]


In [6]:
# ============================================================================
# Step 4: Verification (CRPS and ACC)
# ============================================================================
def crps_ensemble(ens, y):
    out = np.full(ens.shape[0], np.nan)
    finite_ens = np.all(np.isfinite(ens), axis=1)
    mask = finite_ens & np.isfinite(y)
    if not mask.any():
        return out
    e = ens[mask]
    yv = y[mask]
    term1 = np.mean(np.abs(e - yv[:, None]), axis=1)
    diff = np.abs(e[:, :, None] - e[:, None, :])
    term2 = 0.5 * diff.mean(axis=(1, 2))
    out[mask] = term1 - term2
    return out
 
# ---- (b) per-system climatology --------------------------------------------
def anomaly_correlation(fc, obs, fc_clim=None, obs_clim=None):
    """ACC with optionally separate forecast and observation climatologies.
    If clims are None, each is taken as the mean of its own series over the
    (finite) sample -- equivalent to Pearson r."""
    m = np.isfinite(fc) & np.isfinite(obs)
    if m.sum() < 2:
        return np.nan
    fc_m, obs_m = fc[m], obs[m]
    if fc_clim is None:
        fc_clim = float(np.mean(fc_m))
    if obs_clim is None:
        obs_clim = float(np.mean(obs_m))
    fa = fc_m  - fc_clim
    oa = obs_m - obs_clim
    den = np.sqrt(np.sum(fa * fa) * np.sum(oa * oa))
    return float(np.sum(fa * oa) / den) if den > 0 else np.nan
 
verif_rows = []
for var in tqdm(vars_keep, desc='Verification   '):
    y_full = ds_target[var]
    valid_years_avail = y_full['valid_year'].values
 
    x_raw   = ds_input_verif[var].values
    x_dress = ds_dress[var].values
    mu      = ds_calib[var + '_mu'].values
    sigma   = ds_calib[var + '_sigma'].values
 
    for il, lead in enumerate(lead_years):
        valid_yrs = init_years_verif + int(lead)
        y_arr = lookup_target(y_full, valid_yrs, valid_years_avail)
 
        # CRPS
        crps_raw    = crps_ensemble(x_raw[:, il, :],   y_arr)
        crps_dress  = crps_ensemble(x_dress[:, il, :], y_arr)
        crps_closed = crps_gaussian(mu[:, il], sigma[:, il], y_arr)
 
        # ACC with per-system climatology (each centered on its own mean)
        raw_mean   = x_raw[:, il, :].mean(axis=1)
        dress_mean = x_dress[:, il, :].mean(axis=1)
        calib_mean = mu[:, il]
 
        acc_raw   = anomaly_correlation(raw_mean,   y_arr)
        acc_dress = anomaly_correlation(dress_mean, y_arr)
        acc_calib = anomaly_correlation(calib_mean, y_arr)
 
        # Per-system clims (recorded for transparency)
        fin_y      = np.isfinite(y_arr)
        clim_obs   = float(np.nanmean(y_arr))      if fin_y.any() else np.nan
        clim_raw   = float(np.nanmean(raw_mean))   if np.isfinite(raw_mean).any()   else np.nan
        clim_calib = float(np.nanmean(calib_mean)) if np.isfinite(calib_mean).any() else np.nan
 
        mean_raw    = float(np.nanmean(crps_raw))
        mean_dress  = float(np.nanmean(crps_dress))
        mean_closed = float(np.nanmean(crps_closed))
        crpss = (1.0 - mean_closed / mean_raw) if (mean_raw and mean_raw > 0) else np.nan
 
        verif_rows.append({
            'variable':    var,
            'lead_year':   int(lead),
            'n_eff':       int(fin_y.sum()),
            'clim_obs':    clim_obs,
            'clim_raw':    clim_raw,
            'clim_calib':  clim_calib,
            'CRPS_raw':    mean_raw,
            'CRPS_dress':  mean_dress,
            'CRPS_closed': mean_closed,
            'CRPSS':       crpss,
            'ACC_raw':     acc_raw,
            'ACC_dress':   acc_dress,
            'ACC_calib':   acc_calib,
            'b':           float(params_dict[var][il, 1]),
            'fit_ok':      bool(fit_flags[var][il]),
        })
 
df_verif = pd.DataFrame(verif_rows)

Verification   : 100%|██████████| 17/17 [00:04<00:00,  3.78it/s]


In [7]:
save_dir = '/glade/derecho/scratch/ksha/EPRI_data/PP_calib/EMOS/'
ds_dress.to_zarr(save_dir+'Dress_Trend.zarr', mode='w')
ds_calib.to_zarr(save_dir+'Calib_Trend.zarr', mode='w')

In [8]:
print("\n=== EMOS with trend ===")
summary = df_verif.groupby('variable')[['CRPS_raw', 'CRPS_dress', 'CRPSS', 'ACC_raw', 'ACC_calib']].mean().round(3)
print(summary.to_string())


=== EMOS with trend ===
                              CRPS_raw  CRPS_dress  CRPSS  ACC_raw  ACC_calib
variable                                                                     
Fort_Bragg_SPEI_09_mean          0.789       0.760  0.023    0.154      0.263
Fort_Bragg_SPEI_48_mean          1.365       2.112 -0.632    0.227      0.294
Fort_Bragg_SPEI_48_min           1.217       2.407 -1.039    0.260      0.335
Fort_Bragg_simple_T2_1d_max      0.898       0.927 -0.037   -0.151     -0.225
Fort_Bragg_simple_T2_1h_max      1.406       1.244  0.123    0.015     -0.043
Fort_Bragg_simple_T2_30d_max     0.578       0.575  0.009    0.029      0.034
Guam_SPEI_03_mean                0.618       0.372  0.401    0.013      0.113
Guam_SPEI_03_min                 1.987       0.506  0.747   -0.066      0.043
Guam_SPEI_48_mean                1.106       0.610  0.428    0.193      0.401
Guam_SPEI_48_min                 1.611       0.643  0.593    0.188      0.395
Guam_simple_T2_1h_max            0.613 